In [6]:
import pandas as pd

df = pd.read_csv("raw_ingested_articles_cleaned.csv")

print("Rows:", len(df))
print(df.columns.tolist())
df.head(3)

Rows: 62203
['article_id', 'title', 'text', 'published_at', 'url', 'author', 'language', 'source_site', 'source_country', 'categories_raw', 'entity_persons_raw', 'entity_organizations_raw', 'entity_locations_raw', 'has_entity_metadata', 'raw_file_name']


,article_id,title,text,published_at,url,author,language,source_site,source_country,categories_raw,entity_persons_raw,entity_organizations_raw,entity_locations_raw,has_entity_metadata,raw_file_name
0,71e3502a665cb219fbf1180ffa8e3d83e48d0a5f,Philippine Peso Hits Record Lows: Is Now the B...,Dubai: Filipino expats in the UAE are seeing s...,2026-03-19T07:13:00.000+02:00,https://gulfnews.com/business/markets/peso-at-...,Nivetha Dayanand,english,gulfnews.com,AE,"['Economy, Business and Finance']",[],[],"[{'name': 'UAE', 'sentiment': 'none'}, {'name'...",1,article_1.json
1,7e1f5b18780831791cf76ed3ed83620a5dd299dd,Power Assets Earnings: Eyes on Use of UKPN Sal...,Power Assets Holdings Ltd\n00006: XHKG (HKG)\n...,2026-03-19T06:30:00.000+02:00,https://www.morningstar.com/company-reports/14...,Lorraine Tan,english,morningstar.com,US,"['Economy, Business and Finance']",[],"[{'name': 'UK Power Networks', 'sentiment': 'n...",[],1,article_10.json
2,3ba4c8cebb884f3c37afb91d82b3c220b6a16c51,Oil rises after Iran strikes Middle East energ...,"BEIJING: Oil prices rose on Thursday, with ben...",2026-03-19T08:53:00.000+02:00,https://www.arabnews.pk/node/2636893/business-...,@Arab_News,english,arabnews.pk,PK,"['Economy, Business and Finance', 'War, Confli...",[],[],[],0,article_100.json


In [7]:
# Create combined text

In [8]:
df["title"] = df["title"].fillna("").astype(str)
df["text"] = df["text"].fillna("").astype(str)

df["combined_text"] = (df["title"] + " " + df["text"]).str.lower()

In [9]:
df.head(3)

,article_id,title,text,published_at,url,author,language,source_site,source_country,categories_raw,entity_persons_raw,entity_organizations_raw,entity_locations_raw,has_entity_metadata,raw_file_name,combined_text
0,71e3502a665cb219fbf1180ffa8e3d83e48d0a5f,Philippine Peso Hits Record Lows: Is Now the B...,Dubai: Filipino expats in the UAE are seeing s...,2026-03-19T07:13:00.000+02:00,https://gulfnews.com/business/markets/peso-at-...,Nivetha Dayanand,english,gulfnews.com,AE,"['Economy, Business and Finance']",[],[],"[{'name': 'UAE', 'sentiment': 'none'}, {'name'...",1,article_1.json,philippine peso hits record lows: is now the b...
1,7e1f5b18780831791cf76ed3ed83620a5dd299dd,Power Assets Earnings: Eyes on Use of UKPN Sal...,Power Assets Holdings Ltd\n00006: XHKG (HKG)\n...,2026-03-19T06:30:00.000+02:00,https://www.morningstar.com/company-reports/14...,Lorraine Tan,english,morningstar.com,US,"['Economy, Business and Finance']",[],"[{'name': 'UK Power Networks', 'sentiment': 'n...",[],1,article_10.json,power assets earnings: eyes on use of ukpn sal...
2,3ba4c8cebb884f3c37afb91d82b3c220b6a16c51,Oil rises after Iran strikes Middle East energ...,"BEIJING: Oil prices rose on Thursday, with ben...",2026-03-19T08:53:00.000+02:00,https://www.arabnews.pk/node/2636893/business-...,@Arab_News,english,arabnews.pk,PK,"['Economy, Business and Finance', 'War, Confli...",[],[],[],0,article_100.json,oil rises after iran strikes middle east energ...


In [10]:
# Label Keyword List

In [11]:
macro_keywords = [
    "inflation", "deflation", "cpi", "ppi", "gdp", "unemployment",
    "interest rate", "interest rates", "rate cut", "rate cuts",
    "rate hike", "rate hikes", "federal reserve", "fed",
    "central bank", "ecb", "bank of japan", "bank of england",
    "monetary policy", "fiscal policy", "bond yields", "treasury yields",
    "exchange rate", "currency market", "recession", "economic growth",
    "economic slowdown", "stimulus", "tariffs", "trade war"
]

industry_keywords = [
    "banking sector", "diversified banks", "regional banks",
    "financial services", "mortgage finance", "payment processing",
    "consumer staples", "food retail", "drug retail", "broadline retail",
    "specialty retail", "automotive retail", "commercial services",
    "professional services", "data processing", "outsourced services",
    "industrial reits", "office reits", "health care reits",
    "retail reits", "residential reits", "hotel & resort reits",
    "telecom tower reits", "data center reits", "passenger airlines",
    "cargo ground transportation", "passenger ground transportation",
    "marine transportation", "rail transportation", "airport services",
    "semiconductor", "semiconductor industry",
    "automotive industry", "oil and gas sector", "energy sector",
    "real estate sector", "technology sector", "healthcare sector",
    "insurance sector", "telecom sector", "mining sector"
]

In [ ]:
# function to return matched tags

In [12]:
def find_matches(text, keyword_list):
    text = str(text).lower()
    matches = []

    for keyword in keyword_list:
        if keyword in text:
            matches.append(keyword)

    return matches

In [7]:
# Create Rule tag columns

In [13]:
df["macro_rule_tags"] = df["combined_text"].apply(lambda x: find_matches(x, macro_keywords))
df["industry_rule_tags"] = df["combined_text"].apply(lambda x: find_matches(x, industry_keywords))

In [14]:
# Convert Tag list to binary rule flags

In [15]:
df["macro_rule"] = df["macro_rule_tags"].apply(lambda x: 1 if len(x) > 0 else 0)
df["industry_rule"] = df["industry_rule_tags"].apply(lambda x: 1 if len(x) > 0 else 0)

In [16]:
# Build entity tags from metadata

In [17]:
def clean_entity_raw(value):
    if pd.isna(value):
        return []
    
    value = str(value).strip()
    
    if value in ["", "None", "nan", "[]"]:
        return []
    
    return [value]

In [18]:
# Prevent missing value errors

In [19]:
df["entity_persons_raw"] = df["entity_persons_raw"].fillna("")
df["entity_organizations_raw"] = df["entity_organizations_raw"].fillna("")
df["entity_locations_raw"] = df["entity_locations_raw"].fillna("")

In [20]:
# Combine entity tag column

In [21]:
df["entity_rule_tags"] = (
    df["entity_organizations_raw"].astype(str) + " | " +
    df["entity_persons_raw"].astype(str) + " | " +
    df["entity_locations_raw"].astype(str)
)

In [22]:
# if metadata says entity exists: entity rule = 1, else 0

In [23]:
df["entity_rule"] = df["has_entity_metadata"].apply(lambda x: 1 if x == 1 else 0)

In [24]:
# Create needs_llm column to identify which row needs llm to identify for all tags

In [25]:
df["needs_llm"] = (
    (df["macro_rule"] == 0) &
    (df["industry_rule"] == 0) &
    (df["entity_rule"] == 0)
).astype(int)

In [26]:
# inspect results

In [27]:
print("Macro rule count:", df["macro_rule"].sum())
print("Industry rule count:", df["industry_rule"].sum())
print("Entity rule count:", df["entity_rule"].sum())
print("Needs LLM count:", df["needs_llm"].sum())

df[[
    "title",
    "macro_rule",
    "macro_rule_tags",
    "industry_rule",
    "industry_rule_tags",
    "entity_rule",
    "entity_rule_tags",
    "needs_llm"
]].head(10)

Macro rule count: 27843
Industry rule count: 5643
Entity rule count: 42805
Needs LLM count: 10813


,title,macro_rule,macro_rule_tags,industry_rule,industry_rule_tags,entity_rule,entity_rule_tags,needs_llm
0,Philippine Peso Hits Record Lows: Is Now the B...,1,"[inflation, ppi, fed, central bank, exchange r...",0,[],1,"[] | [] | [{'name': 'UAE', 'sentiment': 'none'...",0
1,Power Assets Earnings: Eyes on Use of UKPN Sal...,0,[],0,[],1,"[{'name': 'UK Power Networks', 'sentiment': 'n...",0
2,Oil rises after Iran strikes Middle East energ...,1,"[inflation, interest rate, interest rates, fed...",0,[],0,[] | [] | [],0
3,The United States and Japan have agreed to inv...,0,[],1,[energy sector],1,"[{'name': 'Pravda USA', 'sentiment': 'neutral'...",0
4,AIA Group Ltd Has $9.68 Million Position in Th...,1,"[interest rate, interest rates, fed]",0,[],1,"[{'name': 'AIA Group Ltd', 'sentiment': 'negat...",0
5,Academy Capital Management Makes New $12.91 Mi...,0,[],0,[],1,"[{'name': 'Academy Capital Management', 'senti...",0
6,Edison International $EIX Shares Bought by AIA...,0,[],0,[],1,"[{'name': 'Edison International', 'sentiment':...",0
7,"HDFC Bank governance strong, depositors have n...",0,[],0,[],0,[] | [] | [],1
8,"Oman crude hits new high, closes at $153.12",0,[],0,[],1,"[{'name': 'Muscat', 'sentiment': 'negative', '...",0
9,Whale Gains $2.9M from Gold and Silver Price D...,0,[],0,[],1,"[{'name': 'Phemex News', 'sentiment': 'negativ...",0


In [28]:
df.to_csv("prelabelled_dataset.csv", index=False)
print("Saved prelabelled_dataset.csv")

Saved prelabelled_dataset.csv


In [28]:
df.describe()

,has_entity_metadata,macro_rule,industry_rule,entity_rule,needs_llm
count,62203.000000,62203.000000,62203.000000,62203.000000,62203.000000
mean,0.688150,0.447615,0.090719,0.688150,0.173834
std,0.463253,0.497252,0.287212,0.463253,0.378970
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,1.000000,0.000000
75%,1.000000,1.000000,0.000000,1.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000


In [33]:
#df["macro_rule"].value_counts()
df["industry_rule"].value_counts()
#df["entity_rule"].value_counts()
#df["needs_llm"].value_counts()

industry_rule
0    56560
1     5643
Name: count, dtype: int64

In [33]:
sample_unresolved = df[df["needs_llm"] == 1][[
    "title",
    "text",
    "macro_rule_tags",
    "industry_rule_tags",
    "entity_rule_tags"
]].sample(10, random_state=42)

sample_unresolved

,title,text,macro_rule_tags,industry_rule_tags,entity_rule_tags
17103,Pet Furniture Market to Reach US$10.9 Bn by 20...,Pet Furniture Market to Reach US$10.9 Bn by 20...,[],[],[] | [] | []
46446,Erie Insurance receives slight credit downgrad...,Erie Insurance employees celebrate a century o...,[],[],[] | [] | []
52637,SunLive - Tauranga's rates hike: How much more...,Tauranga homeowners will pay an extra $180 to ...,[],[],[] | [] | []
34870,Poddar Pigments Ltd. Stock Hits 52-Week Low Am...,Stock Performance and Market Context\nThe stoc...,[],[],[] | [] | []
30252,Respiratory Drugs Market Forecasted to Reach U...,Authenticated data presented in the Respirator...,[],[],[] | [] | []
1667,Data Patterns among 6 stocks that hit 52-week ...,"On Thursday, the benchmark Sensex rose by 900 ...",[],[],[] | [] | []
39636,Zambia : Economic Woes Have Dampened Christmas...,Economic Woes Have Dampened Christmas Frenzy\n...,[],[],[] | [] | []
29402,"Katsina State to leverage Sukuk, infrastructur...",Nigerian Exchange Group (NGX Group) hosted the...,[],[],[] | [] | []
39144,Markets sink on Trump's 500% tariff threat on ...,"Thursday, 8 January 2026\nSensex drops 663.83 ...",[],[],[] | [] | []
13286,US announces $14.2 million military aid to Leb...,"Hence then, the article about us announces 14 ...",[],[],[] | [] | []


In [35]:
# create needs_llm csv

In [36]:
llm_df = df[df["needs_llm"] == 1][["article_id", "title", "text"]].copy()

print("Rows to send to local LLM:", len(llm_df))
llm_df.head()

Rows to send to local LLM: 10813


,article_id,title,text
7,ee5c979737eabb9aff1b3b9a344ee1a500e3bad0,"HDFC Bank governance strong, depositors have n...",Depositors have no reason to worry as HDFC Ban...
11,7602312d25a54872f135c69c77bf1acadc4e697a,Japan Industrial Output Grows More Than Estima...,-\nJapan Industrial Output Grows More Than Est...
20,5270544ba069185841ca8279d8191bcbee937885,Allworth Financial LP Raises Position in iShar...,Allworth Financial LP increased its stake in i...
23,f0cde1ef2e854e27c7cc37a1e79c6bf0ee145d7b,"VK increases revenue 8% to 160 bln rubles, adj...",19 Mar 2026 10:19 VK increases revenue 8% to 1...
27,994a7da3c3b3bbb8544786982b20bed508ead3a1,India Gold Price Today: Gold Rises Significant...,In a notable market movement reported on April...


In [37]:
llm_df.to_csv("llm_input_dataset.csv", index=False)
print("Saved llm_input_dataset.csv")

Saved llm_input_dataset.csv
